In [1]:
#import zipfile
#with zipfile.ZipFile("mri2.zip", 'r') as zip_ref:
 #   zip_ref.extractall()

In [1]:
import cv2
import os
import numpy as np
import pandas as pd
from math import exp, sqrt
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels
import matplotlib.image as mpimg
from sklearn.utils import shuffle
import keras
import tensorflow as tf
import tensorflow.keras.backend as K
from keras import Input 
from keras.models import Model
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import  BatchNormalization, Concatenate, Add, DepthwiseConv2D, GlobalAveragePooling2D, SeparableConv2D
from tensorflow.keras.optimizers import SGD,RMSprop,Adam,Adagrad
from keras.layers import Conv2D, MaxPooling2D

2024-09-03 08:10:18.677247: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9373] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-03 08:10:18.677298: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-03 08:10:18.678785: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1534] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-09-03 08:10:18.686976: I tensorflow/core/platform/cpu_feature_guard.cc:183] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
data_path = 'MRI2'
data_dir_list = os.listdir(data_path)
img_data_list = []
labels = []

for dataset in data_dir_list:
    img_list = os.listdir(os.path.join(data_path, dataset))
    print('Loaded the images of dataset-{}\n'.format(dataset))
    for img in img_list:
        if img.lower().endswith(('.png', '.jpg', '.jpeg')):
            input_img = cv2.imread(os.path.join(data_path, dataset, img))
            if input_img is not None:
                img = cv2.resize(input_img, (224, 224))
                labels.append(dataset)
                img_data_list.append(img)

label = np.array(labels)
img_data = np.array(img_data_list)
img_data = img_data.astype('float32')
img_data = img_data / 255.0
print(img_data.shape)

Loaded the images of dataset-SMC

Loaded the images of dataset-MCI

Loaded the images of dataset-EMCI

Loaded the images of dataset-CN

Loaded the images of dataset-LMCI

Loaded the images of dataset-AD

(9752, 224, 224, 3)


In [3]:
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

class_names = ['CN', 'SMC', 'LMCI', 'MCI', 'EMCI', 'AD']
class_mapping = {name: idx for idx, name in enumerate(class_names)}
integer_labels = [class_mapping[label] for label in labels]
num_classes = len(class_names)
Y = to_categorical(integer_labels, num_classes)
x, y = shuffle(img_data, Y, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (7801, 224, 224, 3)
X_test shape: (1951, 224, 224, 3)
y_train shape: (7801, 6)
y_test shape: (1951, 6)


In [4]:
X_train = X_train.reshape(-1, 224, 224, 3)
X_test = X_test.reshape(-1, 224, 224, 3)

In [7]:
import keras
from keras.models import Model
from tensorflow.keras import regularizers
from keras.layers import Input, Conv2D, Add, Multiply, Concatenate, AveragePooling2D, Flatten, DepthwiseConv2D, BatchNormalization
from keras.layers import SeparableConv2D, MaxPooling2D, Dense, GlobalAveragePooling2D, Activation
from tensorflow.keras.applications import ResNet50, VGG16, MobileNet

base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:12]:
    layer.trainable = False
x = base_model.output
x1 = Conv2D(64, (3, 3),  dilation_rate = 1, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x1 = BatchNormalization()(x1)
x2 = Conv2D(64, (3, 3),  dilation_rate = 3, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x2 = BatchNormalization()(x2)
x3 = Conv2D(64, (3, 3),  dilation_rate = 5, padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(x)
x3 = BatchNormalization()(x3)

A1 = Add()([x1, x2])
M1 = Multiply()([x2, x3])

x4 = Conv2D(64, (3, 3), padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(A1)
x4 = BatchNormalization()(x4)
g1 = GlobalAveragePooling2D()(x4)
d1 = Dense(1, activation = 'sigmoid')(g1)
M3 = Multiply()([d1, x1])

x5 = Activation('softmax')(M1)
M2 = Multiply()([x3, x5])

C1 = Concatenate()([M2, M3])
x5 = Conv2D(64, (3, 3), padding = 'same', kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01), activation='relu')(C1)
C2 = Concatenate()([x5, x])

x = GlobalAveragePooling2D()(C2)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
x = Dense(32, activation='relu')(x)
output = Dense(6, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='Adagrad', loss='categorical_crossentropy', metrics=['acc'])
model.summary()

2024-09-03 08:11:29.086302: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 4.28MiB (4489216 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2024-09-03 08:11:29.086418: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 4.28MiB (4489216 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2024-09-03 08:11:39.086747: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 4.28MiB (4489216 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2024-09-03 08:11:39.086887: I external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:1101] failed to allocate 4.28MiB (4489216 bytes) from device: CUDA_ERROR_OUT_OF_MEMORY: out of memory
2024-09-03 08:11:39.086897: W external/local_tsl/tsl/framework/bfc_allocator.cc:485] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.00MiB (rounded to 1048576)requested by op AddV2
If the cause is memory fragmentation maybe

ResourceExhaustedError: {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:AddV2] name: 

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint
early_stopping = EarlyStopping(
                              patience=30,
                              min_delta=0.001,
                              monitor="val_acc",
                              restore_best_weights=True
                              )
# Define the model checkpoint callback to save the best weights
checkpoint = ModelCheckpoint('/workspace/notebooks/Mri - Alzimers/Data /Check/'+'-{epoch:02d}.h5', monitor='val_acc', save_best_only=True) 


history = model.fit( X_train, y_train, batch_size = 32, epochs = 200,
                    validation_data = (X_test , y_test), callbacks=[early_stopping, checkpoint], verbose =1)